"""
Name: AqFood_Supply_Chain_Analyzer.py
Project: Aqueduct Food Tool Supply Chain Analyzer 2026
Description:
  This script reads in a user's supply chain (as coordinates, states, or
  country names) and returns a list of the watersheds or aquifers the user
  operates in along with the associates Aqueduct and SBTN data. 
Created: May 2026
Author: Liz Saccoccia(elizabeth.saccoccia@wri.org). Adapted by Carlos 
Original script: 
http://github.com/resource-watch/aqueduct-analysis-microservice/blob/dev/aqueduct/services/food_supply_chain_service.py 
"""

In [1]:
#imports
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import warnings
import numpy as np
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [2]:
#import files
df_aq = gpd.read_file('aq_data_supplychain.geojson', encoding="utf-8-sig")
#add in SBTN Data - this will be updated
SBTN = pd.read_csv('SBTN_SON_V2.csv')
#import production by pfaf
prod = pd.read_csv('all_crops_pfaf_melt.csv')

In [3]:
# Define whether location will use point + radius, state, or country to
# select watersheds
def find_selection_type(row):
    """
    :param row: individual row
    :return: location type (point, state, country, none)
    """
    # If coordiantes exist, location type is point
    if isinstance(row["Latitude"], float) and np.isnan(row["Latitude"]) == False:
        select_by = "point"
    # If state exists WITH country name, location type is state
    elif (isinstance(row["State"], str) == True) & (isinstance(row["Country"], str) == True):
        select_by = "state"
    # If neither of those are true, and a country name exists, location type is country
    elif isinstance(row["Country"], str) == True:
        select_by = "country"
    # Else, no location type given. These will be dropped from the analysis
    else:
        select_by = np.nan
    return select_by
        
# Clean up Radius buffer values. Remove 0's, convert to decimal degrees
# (units of the analysis)
def clean_buffer(row):
    """
    :param row: individual row (1 coordinate + material type)
    :return: buffer radius in decimal degrees
    """
    val = row.Radius  # Find the radius value
    unit = str(row["Radius Units"]).lower()  # Find the radius units
    try:
        float_val = float(val)  # Turn value to floast
        if float_val == 0.0:  # If radius is 0, set to NA
            new_val = np.nan
        elif unit in [
            "miles",
            "mile",]:  
            # If units are in miles, convert to KM (multiple by 1.609), then to degrees (divide by 111)
            new_val = float_val * 1.609 / 111.0
        elif unit in [
            "m",
            "met",
            "meter",
            "meters",
        ]:  # If units are in meters, convert to KM then to degrees (divide by 111)
            new_val = (float_val / 1000) / 111.0
        elif unit in [
            "km",
            "kilometer",
            "kilometers",
        ]:  # If units are in kilometers, convert to degrees (divide by 111)
            new_val = float_val / 111.0
        else:  # Else, return Null radius, report as error
            new_val = np.nan
    except:
        new_val = np.nan
    return new_val
    
# Create buffer (in decimal degrees) around point
def buffer(row):
    """
    :param row: individual row (1 coordinate + material type)
    :return: circle polygon
    """
    return row.geometry.buffer(row.Buffer)

In [4]:
#User input variables

#these need to change based on users data
#Lat/Longs
# Latitude =  #required

# Longitude =  #required

# Radius =  #optional

# Crop =  #required

# Irrigation =  #required

# Volume =  #optional

# #Country + State

# Country = #required

# State =  #optional

# Irrigation = #required

# Volume = #optional



In [5]:
# or table input
df = pd.read_excel('AqueductFood_SupplyChain_Input_Template_All_Data_Levels.xlsm', sheet_name = 'data_entry',skiprows=1)
df = df.reset_index()
df = df.rename(columns={'index':'UniqueID'})
print('data file imported')
df

data file imported


,UniqueID,Business Unit,Latitude,Longitude,Radius,Radius Units,Country,iso_code,State,Commodity,commodity_code,Total Volume,Volume Units,Irrigation
0,0,North America Snacks,38.898992,-77.007986,100.0,kilometer,NaN,NaN,NaN,Wheat,WHEA,450,metric tons,Rainfed
1,1,EU Snacks,-23.568232,-46.693983,20.0,kilometer,NaN,NaN,NaN,Wheat,WHEA,100,kilograms,Both
2,2,North America Snacks,NaN,NaN,NaN,NaN,United States,USA,California,Maize,MAIZ,500,NaN,Irrigated
3,3,South America Snacks,NaN,NaN,NaN,NaN,Argentina,ARG,NaN,Soybean,SOYB,1000,metric tons,Unknown


In [6]:
 # FIND LOCATIONS BASED ON WATER UNIT
            # ----------------------------------
            # Categorize location type
df["Select_By"] = df.apply(lambda x: find_selection_type(x), axis=1)
df

,UniqueID,Business Unit,Latitude,Longitude,Radius,Radius Units,Country,iso_code,State,Commodity,commodity_code,Total Volume,Volume Units,Irrigation,Select_By
0,0,North America Snacks,38.898992,-77.007986,100.0,kilometer,NaN,NaN,NaN,Wheat,WHEA,450,metric tons,Rainfed,point
1,1,EU Snacks,-23.568232,-46.693983,20.0,kilometer,NaN,NaN,NaN,Wheat,WHEA,100,kilograms,Both,point
2,2,North America Snacks,NaN,NaN,NaN,NaN,United States,USA,California,Maize,MAIZ,500,NaN,Irrigated,state
3,3,South America Snacks,NaN,NaN,NaN,NaN,Argentina,ARG,NaN,Soybean,SOYB,1000,metric tons,Unknown,country


In [7]:
#Step 1. Add data and create buffer for points
# SELECT POINT LOCATIONS
df_points = df[df["Select_By"] == "point"]
print("There are {} points".format(len(df_points)))

df_points = df[df['Latitude'].notna()]
# select lat long
# Make sure coordinates are floats. Drop rows where encoding fails
df_points["Latitude"] = pd.to_numeric(df_points["Latitude"], errors="coerce")  # Make sure latitudes are floats
df_points["Longitude"] = pd.to_numeric(df_points["Longitude"], errors="coerce")  # Make sure latitudes are floats

# Create XY from coordinates
df_points["geometry"] = df_points.apply(lambda row: Point(float(row.Longitude), row.Latitude), axis=1)
df_points_check = df_points

# Convert Radius into decimal degree value
df_points["Radius"] = df_points.apply(lambda x: clean_buffer(x), axis=1)

buffered = df_points.filter(["Radius", "geometry", "UniqueID"])
buffered["geometry"] = buffered.apply(lambda x: x.geometry.buffer(x.Radius), axis=1)
buffered = gpd.GeoDataFrame(buffered, geometry=buffered.geometry)
print("The points buffer is complete")
buffered

There are 2 points
The points buffer is complete


,Radius,geometry,UniqueID
0,0.900901,"POLYGON ((-76.10709 38.89899, -76.11142 38.810...",0
1,0.180180,"POLYGON ((-46.51380 -23.56823, -46.51467 -23.5...",1


In [8]:
df_points = df_points.drop(columns='geometry')
df_points_buff = df_points.merge(buffered, on = 'UniqueID')

In [9]:
#df_points_buff = gpd.GeoDataFrame(df_points_buff , crs="EPSG:4326")

In [10]:
df_points_aq = gpd.sjoin(df_points_buff, df_aq, how="left", op="intersects")
df_points_aq["aq_geometry"] = df_points_aq["geometry"]
df_points_aq = df_points_aq.set_geometry("aq_geometry")
df_points_aq = df_points_aq.drop(columns={'geometry'})
df_points_aq = df_points_aq.set_geometry("aq_geometry")
df_points_aq = df_points_aq.rename(columns={'aq_geometry':'geometry'})
df_points_aq = df_points_aq.set_geometry("geometry", crs="EPSG:4326")

ValueError: 'left_df' should be GeoDataFrame, got <class 'pandas.core.frame.DataFrame'>

In [ ]:
df_states= df[df["Select_By"] == "state"]
print("There are {} states".format(len(df_states)))

In [ ]:
#match aq to input
df_aq = df_aq.rename(columns={"gid_0": "iso_code", "name_0": "Country", "name_1": "State"})

In [ ]:
df_states_aq = df_states.merge(df_aq, on = ["iso_code", "Country", "State"], how = 'left')
df_states_aq


In [ ]:
df_countries= df[df["Select_By"] == "country"]
print("There are {} Countries".format(len(df_countries)))

In [ ]:
df_countries_aq = df_countries.merge(df_aq, on = ["iso_code", "Country"], how = 'left')
df_countries_aq
admin_aq = pd.concat([df_states_aq, df_countries_aq, df_points_aq], ignore_index=True)
admin_aq

In [ ]:
admin_aq = gpd.GeoDataFrame(admin_aq, crs="EPSG:4326")

basins = admin_aq.dissolve(by=['UniqueID','pfaf_id'])
basins = basins.reset_index()
basins= basins[['UniqueID',	'pfaf_id', 'geometry', 'Business Unit',	'Latitude', 'Longitude', 'Radius',	'Radius Units', 'Country', 'iso_code',
            'State','Commodity', 'commodity_code', 'Total Volume', 'Volume Units','Irrigation', 'bws_raw','bws_score', 'bws_cat', 'bws_label']]
#Clean up basins
#remove pfafid = -9999
basins = basins[basins['pfaf_id'] != -9999]
basins = basins.dropna(subset=['pfaf_id'])
basins
print("There are {} basins found".format(len(basins)))

In [ ]:
#basins.to_file('data_test.geojson')

In [ ]:
#merge SBTN and production to data
basins = basins.merge(SBTN, on = ['pfaf_id'])
basins = basins.merge(prod, on = ['pfaf_id','commodity_code','Irrigation'], how = 'left')
basins

In [ ]:
# calculate proportion sourced from each basin
pro_per_entry = basins.groupby('UniqueID').sum('Basin Production')
pro_per_entry  = pro_per_entry.reset_index()
pro_per_entry = pro_per_entry.rename(columns={'Basin Production':"summed production"})
pro_per_entry = pro_per_entry[['UniqueID',"summed production"]]
pro_per_entry

In [ ]:
basins = basins.merge(pro_per_entry, on = 'UniqueID')
basins['Production Sourced From Basin'] = basins['Total Volume']*(basins['Basin Production']/basins['summed production'])
basins

In [ ]:
Basin_results = basins[basins['Production Sourced From Basin'].notna()]
Basin_results = Basin_results.drop(columns={'Unnamed: 0','commodity_code','Basin Production', 'summed production'})
Basin_results